# LC 297 — Serialize and Deserialize Binary Tree

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> BFS level-order with explicit
"null" markers fully captures a tree's structure as a flat
string. Deserialization replays BFS using the same order —
each non-null node pops two tokens from the queue as its
left and right children.
</div>

## Official Problem Statement

Design an algorithm to serialize and deserialize a binary tree.
There is no restriction on how your algorithm works as long as
a tree can be serialized to a string and the string deserialized
to the original tree structure.

**Example 1:**
```
Input:  root = [1,2,3,null,null,4,5]
Output: [1,2,3,null,null,4,5]
```
**Example 2:**
```
Input:  root = []
Output: []
```
**Constraints:**
- Number of nodes: `0 to 10^4`
- `-1000 <= Node.val <= 1000`

## What This Is Actually Asking

Convert a binary tree to and from a string (or any flat format)
so the round trip produces the identical tree structure and
values.

The challenge is representing null children unambiguously.
Without null markers, you can't distinguish a left-skewed tree
from a right-skewed one using just the values.

BFS level-order naturally places nodes in parent-child order.
Adding "null" for missing children gives enough information to
reconstruct the tree without ambiguity in a second BFS pass.

## Walk Through an Example by Hand

```
Tree:        1
            / \
           2   3
              / \
             4   5

SERIALIZE (BFS):
  Queue: [1]
  Pop 1 → tokens: "1", push 2, push 3
  Queue: [2, 3]
  Pop 2 → tokens: "2", no children → push null,null
  Pop 3 → tokens: "3", push 4, push 5
  Queue: [null, null, 4, 5]
  Pop null → tokens: "null"
  Pop null → tokens: "null"
  Pop 4   → tokens: "4", no children
  Pop 5   → tokens: "5", no children

  Result: "1,2,3,null,null,4,5,null,null,null,null"

DESERIALIZE:
  tokens = ["1","2","3","null","null","4","5",...]
  root = TreeNode(1), node_queue = [root], i = 1
  Pop node(1): left=TreeNode("2"), right=TreeNode("3")
  Pop node(2): left=null, right=null
  Pop node(3): left=TreeNode("4"), right=TreeNode("5")
  → reconstructed tree ✓
```

## The Picture

```
Tree:          1
              / \
             2   3
                / \
               4   5

BFS token stream (left to right, level by level):

  Level 0:  [1]
  Level 1:  [2, 3]
  Level 2:  [null, null, 4, 5]
  Level 3:  [null, null, null, null]

  String: "1,2,3,null,null,4,5"
           (trailing nulls can be trimmed)

DESERIALIZE — reading the token stream:

  i=0  root=1
  i=1  node(1).left  = 2
  i=2  node(1).right = 3
  i=3  node(2).left  = null  (skip)
  i=4  node(2).right = null  (skip)
  i=5  node(3).left  = 4
  i=6  node(3).right = 5

  Key: node queue drives which parent gets which token.
```

## When To Use This Pattern

- When a tree must be stored or transmitted as text, think
  **BFS with null markers**.
- When you need to reconstruct a tree from a flat sequence,
  think **replay the same traversal order with a queue**.
- When preserving structural information (not just values),
  think **explicit null markers** — values alone are ambiguous.
- When a design problem asks for encode/decode symmetry, think
  **same traversal for both directions**.
- When the tree is wide (many nodes per level), BFS is natural;
  for deep trees, preorder DFS is a simpler recursive option.

## The Approach

**Serialize:** BFS the tree. For each node dequeued, append its
value (or "null") to the token list. For non-null nodes, enqueue
left and right children (even if null). Join with commas.

**Deserialize:** Split the string by commas. Create the root from
token[0]. Use a BFS queue of nodes. For each dequeued node,
consume the next two tokens as its left and right children,
creating new nodes (or None) accordingly.

In [ ]:
# Imports + TreeNode helpers
from collections import deque as _dq

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def make_tree(vals):
    if not vals: return None
    root = TreeNode(vals[0]); q = _dq([root]); i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    if not root: return []
    r = []; q = _dq([root])
    while q:
        n = q.popleft()
        if n:
            r.append(n.val)
            q.append(n.left)
            q.append(n.right)
        else:
            r.append(None)
    while r and r[-1] is None: r.pop()
    return r

In [ ]:
# ----------------------------------------------------------
# Harness — round-trip test
# ----------------------------------------------------------
def test_harness(codec_cls):
    cases = [
        [1, 2, 3, None, None, 4, 5],
        [],
        [1],
        [1, 2],
        [1, None, 2, None, 3],
    ]
    passed = 0
    for vals in cases:
        codec = codec_cls()
        original = make_tree(vals)
        data = codec.serialize(original)
        restored = codec.deserialize(data)
        result = tree_to_list(restored)
        ok = result == vals
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | input={vals}"
                  f" | got={result}")
        else:
            print(f"{status} | {vals} → '{data}'"
                  f" → {result}")
        passed += ok
    print(f"\n{passed}/{len(cases)} tests passed")

In [ ]:
from collections import deque

class Codec:
    """
    LC 297 — Serialize and Deserialize Binary Tree

    Serialize: BFS, append val or 'null', join with ','
    Deserialize: split by ',', BFS reconstruction
      - root = tokens[0]
      - for each node in queue: consume next 2 tokens
        as left and right children

    Time:  O(n) both
    Space: O(n) for queue and token list
    """

    def serialize(self, root) -> str:
        """
        Encodes a tree to a single string.
        """
        pass
        # Debug: print(f"tokens={tokens}")

    def deserialize(self, data: str):
        """
        Decodes a string to a tree.
        """
        pass
        # Debug: print(f"i={i} token={tokens[i]}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(Codec)

## Complexity

| Approach | Serialize | Deserialize | Space |
|---|---|---|---|
| BFS with null markers | O(n) | O(n) | O(n) |
| Preorder DFS + null | O(n) | O(n) | O(h) stack |
| Inorder+Preorder | O(n) | O(n log n) | O(n) — no nulls needed |
| JSON / pickle | O(n) | O(n) | O(n) — not interview-friendly |

## Real World Connection

**AWS / Data Engineering context:** Tree serialization is the
foundation of distributed data formats. Apache Parquet and
Arrow encode nested schemas (which are trees) as flat byte
streams — exactly the same null-marker BFS idea at scale.

In financial systems at Citi, expression trees representing
pricing formulas must be serialized to/from databases and
message queues. A robust round-trip codec is critical for
reproducibility of risk calculations.

In distributed systems, serialized tree structures travel over
the wire as protocol buffer messages — the flatten-and-
reconstruct pattern maps directly to protobuf's encoding of
recursive message types.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra